# VoiceGuard — Kaggle training notebook

Run tonight on a Kaggle GPU runtime. Steps:
1. Clone the repo (or upload it as a Kaggle dataset/utility script) so `config.py` and `src/` are importable.
2. Attach the ASVspoof 2019 LA (+ 2021 DF / In-the-Wild) datasets via **Add Data** in the Kaggle sidebar — zero download to disk.
3. Train, evaluate EER on the unseen set, generate demo clips, save everything under `artifacts/` and `demo_clips/`.
4. Download `artifacts/` + `demo_clips/` and upload to Google Drive (see Step 10 of the runbook).

In [ ]:
# --- 1. Get the code onto the Kaggle runtime ---
# Option A: clone from GitHub (after Step 1 push)
!git clone https://github.com/<your-username>/voiceguard.git /kaggle/working/voiceguard
%cd /kaggle/working/voiceguard

In [ ]:
# --- 2. Install deps ---
!pip install -q -r requirements_cuda.txt

In [ ]:
# --- 3. Confirm GPU + device selection ---
import sys; sys.path.insert(0, '/kaggle/working/voiceguard')
from src.device import get_device
print('device:', get_device())
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- 4. Locate attached datasets ---
# After using 'Add Data' in the Kaggle sidebar to attach ASVspoof 2019 LA
# (and optionally 2021 DF / In-the-Wild), list what actually landed under
# /kaggle/input so we can point --protocol / --audio-dir at the right paths.
!ls /kaggle/input
!find /kaggle/input -maxdepth 3 -iname '*.txt' | head -20
!find /kaggle/input -maxdepth 3 -type d | head -40

In [ ]:
# --- 5. Train (Step 5) ---
# Fill in the real protocol/audio paths found above.
PROTOCOL = '/kaggle/input/asvspoof-2019-la/.../ASVspoof2019.LA.cm.train.trn.txt'
AUDIO_DIR = '/kaggle/input/asvspoof-2019-la/.../ASVspoof2019_LA_train/flac'
VAL_PROTOCOL = '/kaggle/input/asvspoof-2019-la/.../ASVspoof2019.LA.cm.dev.trl.txt'
VAL_AUDIO_DIR = '/kaggle/input/asvspoof-2019-la/.../ASVspoof2019_LA_dev/flac'

!python -m src.train \
  --protocol "$PROTOCOL" --audio-dir "$AUDIO_DIR" \
  --val-protocol "$VAL_PROTOCOL" --val-audio-dir "$VAL_AUDIO_DIR" \
  --epochs 3

In [ ]:
# --- 6. Cross-dataset EER on an UNSEEN set (Step 6) ---
# Use ASVspoof 2021 DF or In-the-Wild -- whichever is attached. This is the
# headline credibility number for the pitch; do not report train-set accuracy.
UNSEEN_PROTOCOL = '/kaggle/input/<unseen-dataset>/protocol.txt'
UNSEEN_AUDIO_DIR = '/kaggle/input/<unseen-dataset>/audio'

!python -m src.eval_eer \
  --protocol "$UNSEEN_PROTOCOL" --audio-dir "$UNSEEN_AUDIO_DIR" \
  --dataset-name 'ASVspoof2021-DF-or-InTheWild'

!cat artifacts/metrics.json

In [ ]:
# --- 7. Sanity-check streaming inference on CPU-equivalent path (Step 7) ---
# Run the same RiskEngine the Mac will run tomorrow, against one bonafide
# and one spoof clip from the unseen set, to eyeball low vs high risk.
!python -m src.infer /kaggle/input/<unseen-dataset>/some_bonafide_clip.wav
!python -m src.infer /kaggle/input/<unseen-dataset>/some_spoof_clip.wav

## 8. Generate the demo clip pack (Step 8)
Upload a 10-20s clean reference sample of a teammate as `teammate_ref.wav`
into `/kaggle/working/voiceguard/demo_clips/` (via Kaggle's file upload) before
running this cell.

In [ ]:
import os
os.makedirs('demo_clips', exist_ok=True)

from TTS.api import TTS
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2')

REF = 'demo_clips/teammate_ref.wav'

tts.tts_to_file(
    text='Hi, please transfer two lakh rupees to this account urgently.',
    speaker_wav=REF, language='en', file_path='demo_clips/fraud_en.wav')

tts.tts_to_file(
    text='\u0928\u092e\u0938\u094d\u0924\u0947, \u0915\u0943\u092a\u092f\u093e \u0907\u0938 \u0916\u093e\u0924\u0947 \u092e\u0947\u0902 \u0924\u0941\u0930\u0902\u0924 \u0926\u094b \u0932\u093e\u0916 \u0930\u0941\u092a\u092f\u0947 \u092d\u0947\u091c\u093f\u090f\u0964',
    speaker_wav=REF, language='hi', file_path='demo_clips/fraud_hi.wav')

print('Generated:', os.listdir('demo_clips'))

In [ ]:
# --- 8b. Sanity check the generated clips through the RiskEngine ---
# Genuine (REF) should score low; cloned (fraud_en/fraud_hi) should score high.
!python -m src.infer demo_clips/teammate_ref.wav
!python -m src.infer demo_clips/fraud_en.wav
!python -m src.infer demo_clips/fraud_hi.wav

## 9. Package the handoff bundle (Step 10)
Zip `artifacts/` and `demo_clips/`, download from Kaggle, upload to Google Drive.
Also push the code (already cloned from GitHub) if you made local edits in this notebook session.

In [ ]:
!zip -r voiceguard_handoff.zip artifacts demo_clips
print('Ready to download voiceguard_handoff.zip and upload to Google Drive.')